# RL Foundations for Language Models

---

### What Is Reinforcement Learning, and How Does an Autoregressive LLM Fit into an MDP?

---

In supervised learning (like standard Supervised Fine-Tuning / SFT), the model learns from an expert demonstration. For every input $x$, we provide the exact target token sequence $y^*$. The model computes cross-entropy loss against the ground truth and adjusts its weights.

## Reinforcement Learning (RL)

In RL, there is no ground-truth label for every single step. Instead:

- An agent interacts with an environment.
- The agent takes actions.
- The environment changes its state and sends back a scalar signal called a reward $R$.
- The goal is to learn a strategy (a policy, $\pi$) that maximizes cumulative reward over time.

Instead of telling the model “say exactly this word next,” RL tells the model: “try generating a solution; here is a score evaluating how well you solved the task.”

## 2. The Formal Framework: Markov Decision Process (MDP)

Every classical RL problem is formalized mathematically as a Markov Decision Process (MDP). An MDP is defined as a tuple:

$$
\mathcal{M} = (\mathcal{S}, \mathcal{A}, \mathcal{P}, \mathcal{R}, \gamma)
$$

Let’s break down each component conceptually:

- State space ($\mathcal{S}$): the set of all possible configurations or contexts the agent can observe.
- Action space ($\mathcal{A}$): the set of all possible moves or choices the agent can take.
- Transition probability function ($\mathcal{P}$):
  $$
  P(s_{t+1} \mid s_t, a_t)
  $$
  The probability of moving to a new state $s_{t+1}$ given the current state $s_t$ and the action taken $a_t$.
- Reward function ($\mathcal{R}$):
  $$
  R(s_t, a_t)
  $$
  A scalar feedback signal returned by the environment after taking action $a_t$ in state $s_t$.
- Discount factor ($\gamma \in [0, 1]$): a mathematical factor that determines how much the agent values immediate rewards versus future rewards.

## The Markov Property

An environment is Markovian if the future state depends only on the current state and the current action, not on the entire history of past states:

$$
P(s_{t+1} \mid s_t, a_t, s_{t-1}, a_{t-1}, \dots, s_0) = P(s_{t+1} \mid s_t, a_t)
$$

## 3. Mapping an Autoregressive LLM into an MDP

When we apply RL to a large language model (for example, generating an answer to a math problem or writing code), how do the components of an MDP map to text generation?

### Example

Prompt: “Solve 2x + 4 = 10”

- Time step $t=0$
  - State $(s_0)$: ["Solve 2x + 4 = 10"]
  - Action $(a_0)$: "Subtract"
  - Transition: $s_1 = s_0 + a_0$
  - Reward $(r_0)$: 0

- Time step $t=1$
  - State $(s_1)$: ["Solve 2x + 4 = 10", "Subtract"]
  - Action $(a_1)$: " 4"
  - Transition: $s_2 = s_1 + a_1$
  - Reward $(r_1)$: 0

- ...

- Time step $t=T$ (terminal)
  - State $(s_T)$: ["Solve 2x + 4 = 10", "Subtract 4...", "x = 3", "<EOS>"]
  - Reward $(r_T)$: $+1.0$ if correct or $0.0$ if incorrect

## MDP Component Mapping

| Classical RL | LLM generation equivalent |
| --- | --- |
| Agent / policy ($\pi_\theta$) | The LLM parameters $\theta$ outputting a probability distribution over the vocabulary: $\pi_\theta(a_t \mid s_t)$ |
| State ($s_t$) | The entire sequence of tokens up to step $t$: prompt + all generated tokens so far $(x, y_{<t})$ |
| Action ($a_t$) | Selecting the next token $y_t$ from the tokenizer vocabulary $\mathcal{V}$ (e.g., $|\mathcal{V}| \approx 32\text{k} - 128\text{k}$) |
| Transition ($\mathcal{P}$) | Deterministic text concatenation: $s_{t+1} = [s_t, a_t]$ |
| Episode / rollout | One complete sequence generation from prompt to <EOS> or maximum sequence length |
| Reward ($\mathcal{R}$) | Verifier output, rule-based checker, or a reward model score |

## 4. Key Engineering Differences: LLM RL vs. Classical RL

- Massive discrete action space: In classical RL (like Atari or MuJoCo), action spaces are small discrete sets ($|\mathcal{A}| \approx 4\text{–}18$) or low-dimensional continuous vectors. In LLMs, the action space is the entire vocabulary ($|\mathcal{V}| \approx 32,000$ to $150,000+$ discrete choices at every single time step).
- Deterministic, append-only transitions: The transition function $\mathcal{P}(s_{t+1} \mid s_t, a_t)$ is fully deterministic and trivial: $s_{t+1} = s_t \mathbin{\Vert} a_t$. The environment does not have independent dynamic physics; the state is literally the context window.
- Severe reward sparsity: In standard generation (for example, mathematical reasoning), intermediate tokens receive no immediate feedback ($r_t = 0$). Only when the final <EOS> token is emitted and the solution extracted do we receive a non-zero terminal reward $r_T \in \{0, 1\}$.


In this MDP framing of an LLM:

- If a prompt has length $L = 50$ tokens and the model generates a response of length $T = 100$ tokens before emitting <EOS>, how many actions did the agent take?
- What is the exact dimensionality of the policy’s output at step $t = 10$?

## 1. Number of Actions Taken

Your answer: 101 actions (including <EOS>).

**Verdict:** Correct intuition. If the generated response has 100 text tokens followed by 1 <EOS> token, the agent produced a total of 101 tokens. In an MDP, every single generated token is an action sampled from the policy. So the agent took 101 actions (or steps) in that episode/rollout.

## 2. Dimensionality of the Policy Output at Step $t = 10$

Your answer: 60.

**Verdict:** Let’s look at what got mixed up here.

You likely arrived at 60 by adding the prompt length (50) and the current step (10) to get 60. That is the length of the current input sequence/state context ($s_{10}$).

However, policy output dimensionality means: what is the shape or size of the tensor the model produces when it needs to choose an action?

### The mechanism

At step $t = 10$, the input state $s_{10}$ contains 60 tokens (50 prompt tokens + 10 previously generated tokens).

The LLM processes this state and produces raw logits at the final position. A softmax is then applied to convert these logits into a probability distribution over every possible choice in the vocabulary:

$$
\pi_\theta(a \mid s_{10}) = \text{Softmax}(\mathbf{z}_{10})
$$

The model must pick an action $a_{10}$ from the entire vocabulary $\mathcal{V}$.

Therefore, the policy’s output at step $t = 10$ is a 1D probability distribution of size $|\mathcal{V}|$ (the vocabulary size, for example 32,000, 50,257, or 128,256 depending on the tokenizer).


## Summary Checklist for Lesson 1.1

- State ($s_t$): the token sequence so far (context length = $L + t$)
- Policy ($\pi_\theta(a_t \mid s_t)$): distribution over the vocabulary of size $|\mathcal{V}|$
- Action ($a_t$): a single token ID chosen from $\mathcal{V}$
- Rollout / trajectory ($\tau$): the full sequence of $(\text{state}, \text{action}, \text{reward})$ tuples from $t = 0$ to $t = T$


## Returns, Credit Assignment, and Reward Dynamics

Now that we know an LLM generation run is a trajectory of states ($s_t$) and actions ($a_t = \text{token}_t$), we need to understand how the model evaluates its decisions and figures out which tokens were good or bad.

## Returns, Credit Assignment, and Reward Dynamics

Now that we know an LLM generation run is a trajectory of states ($s_t$) and actions ($a_t = \text{token}_t$), we need to understand how the model evaluates its decisions and figures out which tokens were good or bad.

## 1. Cumulative Return ($G_t$)

In RL, the agent does not just try to maximize the immediate reward at step $t$; it aims to maximize the total future reward accumulated from step $t$ until the end of the episode ($T$).

This cumulative sum of future rewards is called the return (denoted as $G_t$):

$$
G_t = r_t + \gamma r_{t+1} + \gamma^2 r_{t+2} + \dots + \gamma^{T-t} r_T = \sum_{k=0}^{T-t} \gamma^k r_{t+k}
$$

### Discount factor

- If $\gamma = 0$, the agent is purely greedy and cares only about the immediate reward $r_t$.
- If $\gamma = 1$, all future rewards are weighted equally with immediate rewards.

In LLM generation for reasoning, math, or coding, we typically set $\gamma = 1.0$ because the terminal answer’s correctness matters equally regardless of how many steps it took.

## 2. The Core Problem: The Credit Assignment Problem

Consider an LLM generating a 300-token solution to a mathematical proof:

- $t=0$: [Prompt]
- $t=140$: the crucial mistake
- $t=300$: terminal

```text
[Prompt] ───> "Step 1..." ───> "Let x = -5 instead of +5" ───> ... ───> Final Answer: Wrong
```

The model generated 300 tokens ($a_0, a_1, \dots, a_{299}$).

- Tokens $0$ to $139$ were logically sound.
- At token $140$, the model made a fatal sign error.
- Tokens $141$ to $299$ followed correctly from the mistake, but the final outcome was wrong ($r_T = 0$).

### Why this is difficult

The credit assignment problem asks: when a single scalar reward $R$ arrives at the very end ($t=T$), which specific token decisions ($a_t$) were responsible for the success or failure?

If we naively penalize or reward every single token equally based solely on $r_T$, we introduce massive variance and slow down learning:

- In a winning rollout ($r_T = 1$), bad intermediate filler tokens get reinforced.
- In a losing rollout ($r_T = 0$), brilliant intermediate reasoning steps get penalized.

## 3. Sparse vs. Dense Rewards

How rewards are delivered dramatically impacts both learning speed and system behavior.

### Sparse reward

```text
Step:    t=0       t=1       t=2      ...      t=T-1      t=T
Reward:   0         0         0                 0         +1.0 (Correct Final Answer)
```

### Dense reward

```text
Step:    t=0       t=1       t=2      ...      t=T-1      t=T
Reward:  +0.1     +0.05     +0.2               +0.1       +1.0 (Reward at every step)
```

| Dimension | Sparse rewards | Dense rewards |
| --- | --- | --- |
| Definition | Reward is given only at the end of the trajectory ($r_t = 0$ for $t < T$, $r_T \in \{0, 1\}$). | Reward is given at intermediate steps (for example, per token, per sentence, per reasoning step). |
| Examples in LLMs | Python unit test passes (1/0), math final answer matches ground truth (1/0). | Process Reward Models (PRMs) scoring each reasoning step, token-level cosine similarity. |
| Advantage | Pure objective truth: hard to game or exploit if the test is strict. | Easier credit assignment: stronger gradient signals; guides the model step-by-step. |
| Engineering risk | Extremely high sample complexity; requires many rollouts to stumble upon a reward. | Reward hacking / exploitation: the model learns to generate tokens that trick the intermediate scorer without solving the problem. |

## 4. Reward Shaping and Reward Hacking

To make sparse environments easier to learn, engineers often attempt reward shaping — adding intermediate heuristic rewards:

$$
R_{\text{total}} = R_{\text{outcome}} + \lambda R_{\text{format}} + \beta R_{\text{length}} + \dots
$$

### The failure mode: reward hacking

A model optimizes the exact mathematical objective given to it, not your human intention.

Real-world failure examples in LLMs:

- Length bias hacking: if you add a small positive reward for generating detailed explanations, the LLM learns to generate endless repetitive rambling text to accumulate length bonuses, ignoring the actual solution.
- Format hacking: if you give a $+0.5$ reward for outputting <think>...</think> tags and $+1.0$ for the right answer, the model might learn to output <think> blocks packed with gibberish just to reliably harvest the $+0.5$ without performing actual reasoning.
- Sycophancy / sycophantic drift: if a learned reward model scores polite, agreeable answers higher, the policy learns to agree with user misconceptions rather than providing factually accurate corrections.


## Check for Understanding

Before we move to Lesson 1.3 (Policy Gradients & On-Policy Dynamics), consider this engineering scenario:

Suppose you are training an LLM to generate Python functions. You define the reward as:

- $+0.2$ if the code syntax is valid (parses with `ast.parse`)
- $+0.3$ if the code imports all necessary libraries
- $+1.0$ if the code passes all hidden unit tests

If the task is very difficult and the model struggles to pass the unit tests early in training, what shortcut or degenerate behavior is the model most likely to learn first?

### Answer

That is the classic reward hacking (or reward gaming) loop.

Because passing the hidden unit tests ($+1.0$) is mathematically hard, the policy discovers the path of least resistance: generating trivial, syntactically valid code blocks with massive import lists (for example, importing 30 standard libraries and defining an empty function or `pass`) to easily harvest the $+0.5$ reward ($0.2 + 0.3$) without attempting to solve the actual problem.
